In [ ]:
from pathlib import Path

import geopandas as gpd
from tqdm.auto import tqdm

In [ ]:
base_path = Path("../../../processed_data/nbs-river-catchment")
out_dir = base_path / "upstream_basins_100"

points = gpd.read_parquet(base_path / "points_to_catchment.parquet")

In [ ]:
# write GPKG copy for visualisation in QGIS
points.to_file(base_path / "points_to_catchment.gpkg")

In [ ]:
# check all points have a corresponding catchment GPKG
def output_exists(row):
    # dem i and j to use in output name and filename
    dem_i, dem_j = row.dem_i, row.dem_j

    base = f"basin_{dem_i}_{dem_j}"
    vect_gpkg = out_dir / f"{base}.gpkg"

    # return early if GPKG exists
    return vect_gpkg.exists()

exists = points.apply(output_exists, axis=1)
points[~exists].reset_index(drop=True)

In [ ]:
# read all catchments - if any have multiple geometries, union_all together
geoms = []
for row in tqdm(points.itertuples(), total=len(points)):
    # dem i and j to use in output name and filename
    dem_i, dem_j = row.dem_i, row.dem_j

    base = f"basin_{dem_i}_{dem_j}"
    vect_gpkg = out_dir / f"{base}.gpkg"
    try:
        geom = gpd.read_file(vect_gpkg).geometry
    except:
        print(vect_gpkg)
    geoms.append(geom.union_all())

In [ ]:
# set up catchments GeoDataFrame as a copy of the outlet points
# replaced by the corresponding (Multi)Polygon catchment geometries
catchments = points.copy()
catchments.geometry = geoms
catchments

In [ ]:
# sense check any MultiPolygon catchments (should all be cases where parts touch at grid cell corners)
catchments[(catchments.geometry.geom_type != 'Polygon')].iloc[0].geometry

In [ ]:
# sense check any single-pixel catchments
catchments[(catchments.geometry.area == 900)]

In [ ]:
# save to Geoparquet
catchments.to_parquet(base_path / "catchments.parquet")

In [ ]:
# save to GPKG for QGIS visualisation
catchments.to_file(base_path / "catchments.gpkg")